In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
data=pd.read_csv('/kaggle/input/red-wine-quality-cortez-et-al-2009/winequality-red.csv')

# Check data samples
Let's look at a few data samples with head() method.

In [ ]:
data.head()

In [ ]:
data.shape

# Features

| Feature           | Significance  |
| ----------------- | ------------- |
| Fixed acidity     | Most acids involved with wine or fixed or nonvolatile (do not evaporate readily)  |
| Volitile acidity  | The amount of acetic acid in wine, which at too high of levels can lead to an unpleasant, vinegar taste  |
| Citric acid       |  Found in small quantities, citric acid can add 'freshness' and flavor to wines     |
| Residual sugar    |  it's rare to find wines with less than 1 gram/liter and wines with greater than 45 grams/liter are considered sweet. |
| Chlorides         |  The amount of salt in the wine. |
| Free sulphur dioxide | The free form of SO2 exists in equilibrium between molecular SO2 (as a dissolved gas) and bisulfite ion; it prevents microbial growth and the oxidation of wine. |
| Total sulphur dioxide | Amount of free and bound forms of S02; in low concentrations, SO2 is mostly undetectable in wine, but at free SO2 concentrations over 50 ppm, SO2 becomes evident in the nose and taste of wine. |
| Density           |  The density of water is close to that of water depending on the percent alcohol and sugar content. |
| pH                |  Describes how acidic or basic a wine is on a scale from 0 (very acidic) to 14 (very basic); most wines are between 3-4 on the pH scale.              |
| Sulphates         | Wine additive which can contribute to sulfur dioxide gas (S02) levels, wich acts as an antimicrobial and antioxidant. |
| Alcohol           | The percentage of alcohol contents in the wine.              |

In [ ]:
feature_list = data.columns[:-1].values
label = [data.columns[-1]]

print ("Feature list:", feature_list)
print ("Label:", label)

# Data statistics
Let's use info() method to get quick description of data.

In [ ]:
data.info()

*  Total entries: 1599 (Tiny dataset by ML standard)
*  There are total 12 columns: 11 features + 1 label
*    Label column: quality
*    Features: [fixed acidity, volitile acidity, citric acid, residual sugar, cholrides, free sulphur dioxide, total sulphur dioxide,      density, pH, sulphates, alcohol]
*  All columns are numeric (float64) and label is an integer.

In [ ]:
data.describe()

* This one prints count and statistical properties - mean, standard deviations and quartiles.

* The wine quality can be between 0 and 10, but in this dataset, the quality values are between 3 and 8. Let's look at the distribution of examples by the wine quality.

In [ ]:
data['quality'].value_counts()

* High quality value --> better quality of wine
* You can see that there are lots of samples of average wines than good or the poor quality ones.
* Many examples with quality = 5 or 6

# The information can be viewed through histogram plot.
* 
* A Histogram gives the count of how many samples occurs within a specific range (bins).
* The x-axis denotes the range of values in a feature and
* The y-axis denotes the frequency of samples with those specific values.

In [ ]:
sns.set()

In [ ]:
data.quality.hist()
plt.xlabel('Wine Quality')
plt.ylabel('Count')

Note taller bars for quality 5 and 6 compared to the

In a similar manner, we can plot all numerical attributes with histogram plot for quick examination.

In [ ]:
data.hist(bins=50,figsize=(15,15))
# display histogram
plt.show()

### A few observations based on these plots: 
* Features are at different scales.
* Features have different distributions -
* A few are tail heavy. e.g. residual sugar, free so2
* A few are multiple modes. e.g. volitile acidity, citric acid

# Create test set

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_set, test_set = train_test_split(data, test_size=0.2, random_state=42)

# Stratified sampling
* Data distribution may not be uniform in real world data.
* Random sampling - by its nature - introduces biases in such data sets.

In [ ]:
data.quality.hist()
plt.xlabel('Wine Quality')
plt.ylabel('Count')

* Many examples of class 5 and 6 compared to the other classes.
* This causes a problem while random sampling. The test distribution may not match with the overall distribution.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in sss.split(data, data["quality"]):
  strat_train_set = data.loc[train_index]
  strat_test_set = data.loc[test_index]

In [ ]:
train_index, test_index= next(sss.split(data, data["quality"]))
strat_train_set = data.loc[train_index]
strat_test_set = data.loc[test_index]

In [ ]:
strat_train_set

Let's examine the test set distribution by the wine quality that was used for stratified sampling.

In [ ]:
strat_dist = strat_test_set["quality"].value_counts() / len(strat_test_set)

Now compare this with the overall distribution:

In [ ]:
overall_dist = data["quality"].value_counts() / len(data)

Let's look at them side-by-side:

In [ ]:
dist_comparison = pd.DataFrame({'overall': overall_dist, 'stratified': strat_dist})
dist_comparison['diff(s-o)'] = dist_comparison['stratified'] - dist_comparison['overall']
dist_comparison['diff(s-o)_pct'] = 100*(dist_comparison['diff(s-o)']/dist_comparison['overall'])

In [ ]:
dist_comparison

Let's contrast this with random sampling:

In [ ]:
random_dist = test_set["quality"].value_counts() / len(test_set)
random_dist

In [ ]:
dist_comparison['random'] = random_dist
dist_comparison['diff(r-o)'] = dist_comparison['random'] - dist_comparison['overall']
dist_comparison['diff(r-o)_pct'] = 100*(dist_comparison['diff(r-o)']/dist_comparison['overall'])

# Sampling bias comparison
* Compare the difference in distribution of stratified and uniform sampling:
 
* Stratified sampling gives us test distribution closer to the overall distribution than the random sampling.

In [ ]:
dist_comparison.loc[:, ['diff(s-o)_pct', 'diff(r-o)_pct']]

# Step 3: Data visualization

* Performed on training set.
* In case of large training set -
  * Sample examples to form **exploration set**.
* Enables to understand features and their relationship among themselves and with output label.

In [ ]:
exploration_set = strat_train_set.copy()

In [ ]:
sns.scatterplot(x='fixed acidity', y='density', hue='quality',
                data=exploration_set)

In [ ]:
exploration_set.plot(kind='scatter', x='fixed acidity', y='density', alpha=0.5,
                     c="quality", cmap=plt.get_cmap("jet"))

In [ ]:
corr_matrix = exploration_set.corr()

In [ ]:
corr_matrix['quality']

Notice that quality has strong positive correlation with alcohol content [0.48] and strong negative correlation with volitile acidity [-0.38].

In [ ]:
plt.figure(figsize=(14,7))
sns.heatmap(corr_matrix, annot=True)

In [ ]:
import warnings

# Suppress specific warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")

attribute_list = ['citric acid', 'pH', 'alcohol', 'sulphates', 'quality']
sns.pairplot(exploration_set[attribute_list])
plt.show()

In [ ]:
from pandas.plotting import scatter_matrix
scatter_matrix(exploration_set[attribute_list])
plt.show()

# Separate features and labels from the training set.

In [ ]:
wine_features = strat_train_set.drop("quality", axis=1)

wine_labels = strat_train_set['quality'].copy()

In [ ]:
wine_features.isna().sum()

* `SimpleImputer` class for filling up missing values with. say, `median` value.
* The `strategy` contains instructions as how to replace the missing values.  In this case, we specify that the missing value should be replaced by the median value.  


In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")

In [ ]:
imputer.fit(wine_features)

In [ ]:
imputer.statistics_

Note that these are median values for each feature.  We can cross-check it by calculating median on the feature set:

In [ ]:
wine_features.median()

In [ ]:
tr_features = imputer.transform(wine_features)

In [ ]:
tr_features.shape

# Converting categories to numbers

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
ordinal_encoder = OrdinalEncoder()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
cat_encoder = OneHotEncoder()

# Transformation Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

transform_pipeline = Pipeline([
                               ('imputer', SimpleImputer(strategy="median")),
                               ('std_scaler', StandardScaler()),])
transform_pipeline

In [ ]:
wine_features_tr = transform_pipeline.fit_transform(wine_features)

# Select and train ML model

* It's a good practice to build a quick baseline model on the preprocessed data and get an idea about model performance.

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(wine_features_tr, wine_labels)

In [ ]:
from sklearn.metrics import mean_squared_error

quality_predictions =  lin_reg.predict(wine_features_tr)
mean_squared_error(wine_labels, quality_predictions)

Let's evaluate performance on the test set.

* We need to first apply transformation on the test set and then apply the model prediction function.

In [ ]:
wine_features_test = strat_test_set.drop("quality", axis=1)

wine_labels_test = strat_test_set['quality'].copy()

wine_features_test_tr = transform_pipeline.transform(wine_features_test)

quality_test_predictions = lin_reg.predict(wine_features_test_tr)
mean_squared_error(wine_labels_test, quality_test_predictions)

In [ ]:
plt.scatter(wine_labels_test, quality_test_predictions)
plt.plot(wine_labels_test, wine_labels_test, 'r-')
plt.xlabel('Actual quality')
plt.ylabel('Predicted quality')

The model seem to be making errors on the best and poor quality wines.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor()
tree_reg.fit(wine_features_tr, wine_labels)

In [ ]:
quality_predictions =  tree_reg.predict(wine_features_tr)
mean_squared_error(wine_labels, quality_predictions)

In [ ]:
quality_test_predictions = tree_reg.predict(wine_features_test_tr)
mean_squared_error(wine_labels_test, quality_test_predictions)

**Note that the training error is 0, while the test error is 0.58. This is an example of an overfitted model.**

In [ ]:
plt.scatter(wine_labels_test, quality_test_predictions)
plt.plot(wine_labels_test, wine_labels_test, 'r-')
plt.xlabel('Actual quality')
plt.ylabel('Predicted quality')

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
def display_scores(scores):
  print("Scores:", scores)
  print("Mean:", scores.mean())
  print("Standard deviation:", scores.std())

In [ ]:
scores = cross_val_score(lin_reg, wine_features_tr, wine_labels,
                         scoring="neg_mean_squared_error", cv=10)
lin_reg_mse_scores = -scores
display_scores(lin_reg_mse_scores)

In [ ]:
scores = cross_val_score(tree_reg, wine_features_tr, wine_labels,
                         scoring="neg_mean_squared_error", cv=10)
tree_mse_scores = -scores
display_scores(tree_mse_scores)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = RandomForestRegressor()
forest_reg.fit(wine_features_tr, wine_labels)

scores = cross_val_score(forest_reg, wine_features_tr, wine_labels,
                         scoring="neg_mean_squared_error", cv=10)
forest_mse_scores = -scores
display_scores(forest_mse_scores)

In [ ]:
quality_test_predictions = forest_reg.predict(wine_features_test_tr)
mean_squared_error(wine_labels_test, quality_test_predictions)

In [ ]:
plt.scatter(wine_labels_test, quality_test_predictions)
plt.plot(wine_labels_test, wine_labels_test, 'r-')
plt.xlabel('Actual quality')
plt.ylabel('Predicted quality')

* Random forest looks more promising than the other two 

# Finetuning the  models

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = [
 {'n_estimators': [3, 10, 30], 'max_features': [2, 4, 6, 8]},
 {'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]},
]

In [ ]:
grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)

In [ ]:
grid_search.fit(wine_features_tr, wine_labels)

In [ ]:
 grid_search.best_params_

In [ ]:
cvres = grid_search.cv_results_
for mean_score, params in zip(cvres["mean_test_score"], cvres["params"]):
  print(-mean_score, params)

In [ ]:
 grid_search.best_estimator_

# Randomized Search

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
 feature_importances = grid_search.best_estimator_.feature_importances_

In [ ]:
sorted(zip(feature_importances, feature_list), reverse=True)


#### Evaluation on test set

Now that we have a reasonable model, we evaluate its performance on the test set.

In [ ]:
wine_features_test = strat_test_set.drop("quality", axis=1)

wine_labels_test = strat_test_set['quality'].copy()

wine_features_test_tr = transform_pipeline.transform(wine_features_test)

In [ ]:
quality_test_predictions = grid_search.best_estimator_.predict(
    wine_features_test_tr)

In [ ]:
mean_squared_error(wine_labels_test, quality_test_predictions)

**It's a good idea to get  95%  confidence interval of the evaluation metric. It can be obtained by the following code**

In [ ]:
from scipy import stats
confidence = 0.95
squared_errors = (quality_test_predictions - wine_labels_test) ** 2
stats.t.interval(confidence, len(squared_errors) - 1,
                 loc=squared_errors.mean(),
                 scale=stats.sem(squared_errors))